# EDA: LoL-V2T annotations and caption-derived events

Run `python -m data.scripts.download_lolv2t` first (annotations only are enough here).

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from common.lolv2t import weak_events

root = Path("../data/lolv2t/annotations")
clips = []
for split, f in [("train", "training"), ("val", "validation"), ("test", "testing")]:
    for cid, c in json.loads((root / f"{f}.json").read_text()).items():
        ev = weak_events(c)
        clips.append(
            {
                "split": split,
                "clip": cid,
                "duration": c["duration"],
                "sentences": len(c["sentences"]),
                **{
                    t: sum(e.event_type == t for e in ev)
                    for t in ["kill", "objective", "teamfight"]
                },
            }
        )
df = pd.DataFrame(clips)
df.groupby("split").agg(clips=("clip", "count"), hours=("duration", lambda s: s.sum() / 3600))

## Clip lengths

Windows are 6 s, so clips shorter than that produce no temporal windows.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df.duration.clip(upper=120), bins=60)
ax.set(xlabel="clip duration (s)", ylabel="clips")
print("share of clips under 6 s:", (df.duration < 6).mean())
plt.show()

## Weak event counts

How often the keyword rules fire. Clips with none are pure background for the detector.

In [ ]:
print(df[["kill", "objective", "teamfight"]].sum())
print("clips with no event:", (df[["kill", "objective", "teamfight"]].sum(axis=1) == 0).mean())

## Spot-check the rules

Read a sample: the labels are only as good as these keywords.

In [ ]:
ann = json.loads((root / "training.json").read_text())
random.seed(0)
for cid in random.sample(list(ann), 5):
    for (s, e), t in zip(ann[cid]["timestamps"], ann[cid]["sentences"], strict=True):
        hit = [x.event_type for x in weak_events({"timestamps": [[s, e]], "sentences": [t]})]
        if hit:
            print(f"{cid} {s:5.1f}s {hit}: {t}")